# Task 63: Convolutional neural-network demonstration

Reference: https://www.tensorflow.org/tutorials/images/cnn

Uses TensorFlow documentation because the PDF's CNN introduction URL is inaccessible. Dataset is 8x8 digits, not MNIST.

In [1]:
# Task 63: Convolutional neural-network demonstration
# Reference: https://www.tensorflow.org/tutorials/images/cnn

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
import numpy as np
import tensorflow as tf

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(13)
tf.config.threading.set_intra_op_parallelism_threads(1)
tf.config.threading.set_inter_op_parallelism_threads(1)
tf.config.experimental.enable_op_determinism()

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

# Built-in 8x8 handwritten images allow a quick offline CNN demonstration.
digits = load_digits()
images = (digits.images / 16.0).astype(np.float32)[..., np.newaxis]
x_train, x_test, y_train, y_test = train_test_split(
    images, digits.target, test_size=0.25, stratify=digits.target, random_state=13)
model = tf.keras.Sequential([
    tf.keras.Input(shape=(8, 8, 1)),
    tf.keras.layers.Conv2D(8, (3, 3), padding='same', activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(24, activation='relu'),
    tf.keras.layers.Dense(10)
])
optimizer = tf.keras.optimizers.Adam(learning_rate=0.003)
rng = np.random.default_rng(13)
print('Image shape:', images.shape[1:])
print('Training/test images:', len(x_train), len(x_test))
print('Architecture: Conv2D -> MaxPooling2D -> Flatten -> Dense -> 10 logits')
print('Trainable parameters:', model.count_params())

def train_epoch(model, optimizer, x, y, order, batch_size=64):
    losses = []
    for start in range(0, len(order), batch_size):
        indices = order[start:start + batch_size]
        x_batch = tf.convert_to_tensor(x[indices], dtype=tf.float32)
        y_batch = tf.convert_to_tensor(y[indices], dtype=tf.int64)
        with tf.GradientTape() as tape:
            logits = model(x_batch, training=True)
            loss = tf.reduce_mean(tf.keras.losses.sparse_categorical_crossentropy(
                y_batch, logits, from_logits=True))
        gradients = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(gradients, model.trainable_variables))
        losses.append(float(loss.numpy()))
    return float(np.mean(losses))


def evaluate(model, x, y, batch_size=128):
    logits = []
    for start in range(0, len(x), batch_size):
        logits.append(model(tf.convert_to_tensor(x[start:start + batch_size]),
                            training=False).numpy())
    logits = np.concatenate(logits)
    predictions = np.argmax(logits, axis=1)
    loss = float(tf.reduce_mean(tf.keras.losses.sparse_categorical_crossentropy(
        y, logits, from_logits=True)).numpy())
    accuracy = float(np.mean(predictions == y))
    return loss, accuracy, predictions

for epoch in range(1, 7):
    loss = train_epoch(model, optimizer, x_train, y_train, rng.permutation(len(x_train)))
    print(f'Epoch {epoch}: training loss={loss:.4f}')
loss, accuracy, predictions = evaluate(model, x_test, y_test)
print('Test loss:', round(loss, 4))
print('Test accuracy:', round(accuracy, 4))
print('First ten actual digits:', y_test[:10])
print('First ten predicted digits:', predictions[:10])


Image shape: (8, 8, 1)
Training/test images: 1347 450
Architecture: Conv2D -> MaxPooling2D -> Flatten -> Dense -> 10 logits
Trainable parameters: 3426
Epoch 1: training loss=2.1797
Epoch 2: training loss=1.7931
Epoch 3: training loss=1.2403
Epoch 4: training loss=0.7637
Epoch 5: training loss=0.4938
Epoch 6: training loss=0.3765
Test loss: 0.3646
Test accuracy: 0.9
First ten actual digits: [5 2 2 0 8 5 0 0 0 9]
First ten predicted digits: [5 2 2 0 8 5 0 0 0 8]
